# Coercing Numbers to Strings (`coerce_numbers_to_str`)

As we learned in previous sections, Pydantic deliberately refuses to coerce arbitrary objects (including numbers) into strings by default. This design choice prevents silent data corruption (e.g., accidentally stringifying a complex dictionary when you expected a plain text string).

However, in many real-world datasets, numeric values (like zip codes, phone numbers, or account IDs) are inconsistently passed as integers or floats. Pydantic provides a specific configuration to safely handle this.

## 1. The Default Behavior (No Coercion)

By default, passing a number to a string field raises an error.

```python
from pydantic import BaseModel, ValidationError

class DefaultModel(BaseModel):
    field: str

try:
    m1 = DefaultModel(field=100)
except ValidationError as e:
    print(e)
    # Output: Input should be a valid string

```

## 2. Enabling Numeric Coercion

You can explicitly tell Pydantic to safely cast numbers to strings using the `coerce_numbers_to_str` configuration.

```python
from pydantic import ConfigDict
from decimal import Decimal

class CoerciveModel(BaseModel):
    model_config = ConfigDict(coerce_numbers_to_str=True)
    field: str

# All of these will now successfully instantiate as strings!
m2 = CoerciveModel(field=100)          # Int
m3 = CoerciveModel(field=1.5)          # Float
m4 = CoerciveModel(field=Decimal("10.5")) # Decimal

print(repr(m2.field)) # '100'
print(repr(m3.field)) # '1.5'
print(repr(m4.field)) # '10.5'

```

### Limitations of this Configuration:

This configuration is highly targeted. It **only** applies to standard numeric types (`int`, `float`, `Decimal`).

* If you pass a **Boolean** (`True`), it will fail.
* If you pass a **Complex Number** (`3 + 4j`), it will fail.
* If you pass a **Dictionary** or **List**, it will fail.

## 3. The Strict Mode Override

It is critical to remember that `coerce_numbers_to_str` is a **Lax Mode feature**. If you enable this setting, but you *also* enable Strict Mode, Strict Mode wins.

```python
class ConflictedModel(BaseModel):
    model_config = ConfigDict(
        coerce_numbers_to_str=True, 
        strict=True  # STRICT MODE OVERRIDES LAX FEATURES
    )
    field: str

try:
    m5 = ConflictedModel(field=100)
except ValidationError as e:
    print("Failed! Strict mode blocked the coercion.")

```


### Interview Preparation: Coercing Numbers to Strings

<details>
<summary><b>1. Why does Pydantic have a dedicated `coerce_numbers_to_str` setting, rather than just coercing everything to a string by default?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
In Python, every single object has a string representation (via `__str__` or `__repr__`). If Pydantic auto-coerced everything to a string, it would hide severe API bugs. For example, if an upstream API changed a field from a string to a nested dictionary, Pydantic would just blindly store the stringified dictionary, breaking downstream systems. <br><br>
The `coerce_numbers_to_str` setting provides a safe middle ground. It explicitly allows safe, expected coercions (like turning the integer `42` into the string `"42"`) while continuing to block dangerous coercions (like dict-to-string).
</details>

<details>
<summary><b>2. You configure your model with `model_config = ConfigDict(coerce_numbers_to_str=True)`. You have a field typed as a `str`. If you pass the boolean value `True` into this field, will it be coerced to the string `"True"`?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
No, it will raise a `ValidationError`. <br><br>
Even though `bool` is technically a subclass of `int` in Python, Pydantic's `coerce_numbers_to_str` logic strictly limits this feature to actual numeric representations: `int`, `float`, and `Decimal`. It deliberately excludes booleans and complex numbers.
</details>

<details>
<summary><b>3. A developer on your team submits a PR with a model configured like this: <br>`model_config = ConfigDict(coerce_numbers_to_str=True, strict=True)`. <br>They expect the integer `10` to be coerced into the string `"10"`. Will this work? Why or why not?</b> (Click to reveal)</summary>
<br>
<b>Answer:</b><br>
It will not work. It will raise a `ValidationError`. <br><br>
`coerce_numbers_to_str` relies on Pydantic's Lax Coercion engine. When you enable `strict=True`, it completely bypasses the Lax engine, meaning all type coercion—even explicitly configured ones like this—are ignored. The model will demand a native Python string and reject the integer.
</details>